In [1]:
import pandas as pd
import os 

In [13]:
path = r"C:\Users\eswar\OneDrive\Desktop\Project Folder\data\raw" 
processed_path = r"C:\Users\eswar\OneDrive\Desktop\Project Folder\data\processed" 

In [3]:
datasets = {

    "fund_master": pd.read_csv(os.path.join(path, "01_fund_master.csv")),

    "nav_history": pd.read_csv(os.path.join(path, "02_nav_history.csv")),

    "aum_by_fund_house": pd.read_csv(os.path.join(path, "03_aum_by_fund_house.csv")),

    "monthly_sip_inflows": pd.read_csv(os.path.join(path, "04_monthly_sip_inflows.csv")),

    "category_inflows": pd.read_csv(os.path.join(path, "05_category_inflows.csv")),

    "industry_folio_count": pd.read_csv(os.path.join(path, "06_industry_folio_count.csv")),

    "scheme_performance": pd.read_csv(os.path.join(path, "07_scheme_performance.csv")),

    "investor_transactions": pd.read_csv(os.path.join(path, "08_investor_transactions.csv")),

    "portfolio_holdings": pd.read_csv(os.path.join(path, "09_portfolio_holdings.csv")),

    "benchmark_indices": pd.read_csv(os.path.join(path, "10_benchmark_indices.csv")),

    "axis_nav": pd.read_csv(os.path.join(path, "axis_nav.csv")),

    "hdfc_nav": pd.read_csv(os.path.join(path, "hdfc_nav.csv")),

    "icici_nav": pd.read_csv(os.path.join(path, "icici_nav.csv")),

    "kotak_nav": pd.read_csv(os.path.join(path, "kotak_nav.csv")),

    "nippon_nav": pd.read_csv(os.path.join(path, "nippon_nav.csv")),

    "sbi_nav": pd.read_csv(os.path.join(path, "sbi_nav.csv"))

} 

In [5]:
date_columns = {
    "fund_master": ["launch_date"],
    "nav_history": ["date"],
    "aum_by_fund_house": ["date"],
    "monthly_sip_inflows": ["month"],
    "category_inflows": ["month"],
    "industry_folio_count": ["month"],
    "investor_transactions": ["transaction_date"],
    "benchmark_indices": ["date"],
    "portfolio_holdings": ["portfolio_date"]
}

for df_name, cols in date_columns.items():
    for col in cols:
        datasets[df_name][col] = pd.to_datetime(
            datasets[df_name][col],
            errors="coerce"
        ) 

In [11]:
# clean nav_history.csv
# sort
nav = datasets["nav_history"]

nav = nav.sort_values(
    ["amfi_code","date"]
) 

# remove duplicates 
nav = nav.drop_duplicates(
    subset=["amfi_code","date"]
)

#  check missing nav
nav.isnull().sum()

# verify duplicate values 
nav.duplicated(subset=["amfi_code", "date"]).sum()

# verify invalid NAV values 
(nav["nav"] <= 0).sum() 

0

In [14]:
nav.to_csv(os.path.join(processed_path, "02_nav_history.csv"), index=False) 

In [18]:
# cleaning investor_transactions
transactions = datasets["investor_transactions"].copy() 

# Convert transaction date
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"],
    errors="coerce"
) 
transactions["transaction_date"].isnull().sum() 

0

In [19]:
#check transdaction types
transactions["transaction_type"].value_counts()  

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

In [20]:
# Validate amount
(transactions["amount_inr"] <= 0).sum() 

0

In [21]:
# Check date conversion
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"],
    errors="coerce"
) 
transactions["transaction_date"].isnull().sum() 

0

In [22]:
#check KYC status 
transactions["kyc_status"].value_counts() 

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [26]:
# Check Amount Validation 
(transactions["amount_inr"] <= 0).sum() 

# check Duplicate Rows 
transactions.duplicated().sum() 

0

In [30]:
transactions.to_csv(
    os.path.join(processed_path, "08_investor_transactions.csv"),
    index=False
)  

In [39]:
# clean scheme_performance
performance = datasets["scheme_performance"].copy() 

#check data types
performance.dtypes 

#check missing values in return columns 
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct"
]

performance[return_cols].isnull().sum() 

#Flag anomalous return values
for col in return_cols:
    anomalies = performance[
        (performance[col] < -100) |
        (performance[col] > 100)
    ]

    print(f"\nAnomalies in {col}: {len(anomalies)}")
    display(anomalies[["scheme_name", col]])


#Validate expense ratio
performance[
    (performance["expense_ratio_pct"] < 0.1) |
    (performance["expense_ratio_pct"] > 2.5)
] 


# check duplicates 
performance.duplicated().sum() 




Anomalies in return_1yr_pct: 0


,scheme_name,return_1yr_pct



Anomalies in return_3yr_pct: 0


,scheme_name,return_3yr_pct



Anomalies in return_5yr_pct: 0


,scheme_name,return_5yr_pct



Anomalies in benchmark_3yr_pct: 0


,scheme_name,benchmark_3yr_pct


In [44]:
performance.to_csv(
    os.path.join(processed_path, "07_scheme_performance.csv"),
    index=False
) 